# CP201A Lecture: Navigating ACS Data in Python (live coding)

**Monday, September 14, 2026**

This is the notebook from Monday's lecture. It repeats the Lab 3 workflow one more time, at the scale you will use for Assignment 1: a neighborhood made of census tracts, compared with its city and county, and compared with itself five years earlier.

## Learning objectives

* Pull an ACS table for every tract in a county, then keep only the tracts that make up one neighborhood
* Read a tract-level table with estimates and margins of error side by side
* Compare a neighborhood's tracts with the city and county, using shares rather than counts
* Pull the same table for an earlier five-year period and see what "change over time" looks like in the data
* Name the one step we cannot do yet (combining tracts into a single neighborhood number), which is next week

Nothing in this notebook is due. Lab 4 (Wednesday 9/23 and Friday 9/25) is where you do this for your own tracts and your own tables.

## 0. Setup

Same three cells as the top of every notebook from here on: install the package, import, load your saved key.

In [ ]:
%pip install -q census

In [ ]:
from census import Census
import pandas as pd
import numpy as np
import os

In [ ]:
# This is the same code every notebook uses to load your key (saved in Lab 3).
with open(os.path.expanduser('~/census_key.txt')) as f:
    api_key = f.read().strip()

print('Key loaded. It starts with:', api_key[:4] + '...')

c = Census(key=api_key)

## 1. The variables

Same table as Lab 3: B03002, Hispanic or Latino origin by race. Same dictionary, estimates and margins of error together.

In [ ]:
variables_of_interest = {
    'NAME': 'NAME',
    'GEO_ID': 'GEO_ID',
    'B03002_001E': 'total',
    'B03002_001M': 'total_moe',
    'B03002_003E': 'nh_white',
    'B03002_003M': 'nh_white_moe',
    'B03002_004E': 'nh_black',
    'B03002_004M': 'nh_black_moe',
    'B03002_005E': 'nh_native',
    'B03002_005M': 'nh_native_moe',
    'B03002_006E': 'nh_asian',
    'B03002_006M': 'nh_asian_moe',
    'B03002_007E': 'nh_pi',
    'B03002_007M': 'nh_pi_moe',
    'B03002_008E': 'nh_1other',
    'B03002_008M': 'nh_1other_moe',
    'B03002_009E': 'nh_multi',
    'B03002_009M': 'nh_multi_moe',
    'B03002_012E': 'hispanic',
    'B03002_012M': 'hispanic_moe',
}

# The columns that hold numbers. We will convert these from text every time we pull.
numeric_cols = [col for col in variables_of_interest.values() if col not in ['NAME', 'GEO_ID']]

## 2. A neighborhood is a list of tracts

In Lab 3 we typed five tract codes straight into the API call. Today we do it the way you will do it for Assignment 1: pull **every** tract in the county once, then keep the ones we want. Two advantages. You only need to know the county code, and if one of your tract numbers is wrong, the notebook tells you instead of silently returning fewer rows.

### 2.1 Every tract in Alameda County, 2020 to 2024

In [ ]:
ACS_YEAR = 2024   # 2020 to 2024 5-year estimates, on 2020 census tract boundaries

df_all_tracts = pd.DataFrame(
    c.acs5.get(
        list(variables_of_interest.keys()),
        {'for': 'tract:*', 'in': 'state:06 county:001'},
        year=ACS_YEAR
    )
).rename(columns=variables_of_interest)

print(f'{len(df_all_tracts)} tracts in Alameda County')
df_all_tracts.head()

### 2.2 Keep the West Oakland tracts

Tomorrow's tour is West Oakland, so that is today's example. Defining the neighborhood is a **decision you make**, and one you write down in your Data Notes. Here we borrow a definition that already exists: the 13 tracts the West Oakland Community Action Plan (the AB 617 plan co-authored by WOEIP and the Air District, 2019) used to describe the community. Borrowing a boundary from a plan, an agency, or a community organization is a good habit; it makes your work comparable to theirs, and you can cite it.

Two of those tracts, 9819 and 9820, have numbers starting with 98. The Census Bureau reserves 98xx codes for tracts with almost no residents: here, the Port of Oakland and the former Army Base. They belong in the plan's boundary because that is where the pollution comes from, but they will show tiny populations, and dividing by a total of zero produces `NaN` in the share columns. That is fine; just notice it.

To find your own tract numbers, open data.census.gov, search a table, choose Geography, then Census Tract, and use the map; or open the TIGERweb map viewer. Lab 3, section 2, walks through it.

In [ ]:
NEIGHBORHOOD_NAME = 'West Oakland'

# Tract codes are six digits: four before the decimal point and two after.
# Census Tract 4014 is '401400'; Census Tract 4016.01 would be '401601'.
# Source: West Oakland Community Action Plan (BAAQMD and WOEIP, 2019), which used
# ACS 2013 to 2017 table DP05 for these 13 tracts.
WEST_OAKLAND_TRACTS = ['401400', '401500', '401600', '401700', '401800',
                       '402200', '402400', '402500', '402600', '402700',
                       '410500', '981900', '982000']

df_tracts = df_all_tracts[df_all_tracts['tract'].isin(WEST_OAKLAND_TRACTS)].copy()

# Did every tract we asked for come back? If a number is wrong, or a tract was
# split or renumbered in 2020, it shows up here.
found = set(df_tracts['tract'])
missing = [t for t in WEST_OAKLAND_TRACTS if t not in found]
print(f'Asked for {len(WEST_OAKLAND_TRACTS)} tracts, found {len(found)}.')
print('Missing:', missing if missing else 'none')

df_tracts[['NAME', 'tract', 'total', 'total_moe']]

### 2.3 Convert text to numbers

The API hands back text. Same fix as Lab 3, section 3.5.

In [ ]:
for col in numeric_cols:
    df_tracts[col] = pd.to_numeric(df_tracts[col])

df_tracts.info()

## 3. Reading a tract table: estimates and margins of error together

Before we calculate anything, look at the counts and their margins of error side by side. Every number in the estimate column is a survey estimate, and the column next to it says how far off it could be. For small groups in small places, the margin of error can be as large as the estimate itself.

We keep every category B03002 gives us: white, Black, American Indian and Alaska Native, Asian, Native Hawaiian and Pacific Islander, some other race, two or more races (all non-Hispanic), and Hispanic or Latino of any race. Dropping the small categories is a decision, and the people in them are exactly the ones the census has the hardest time seeing, so look at them first.

We are not going to combine or manipulate the margins of error today. We are going to keep them in view.

In [ ]:
groups = ['nh_white', 'nh_black', 'nh_native', 'nh_asian', 'nh_pi', 'nh_1other', 'nh_multi', 'hispanic']

# Estimates for every group, one row per tract
est_cols = ['NAME', 'total'] + groups
df_tracts[est_cols].sort_values('total', ascending=False)

In [ ]:
# The same table, but the margins of error
moe_cols = ['NAME', 'total_moe'] + [f'{g}_moe' for g in groups]
df_tracts[moe_cols].sort_values('total_moe', ascending=False)

Read the two tables against each other. For the largest groups the MOE is a fraction of the estimate. For the smallest (American Indian and Alaska Native, Pacific Islander, some other race, two or more races) the MOE is often as large as the estimate or larger: the survey is telling you it cannot see that group clearly at the tract scale.

Two questions to hold: in which tract is the Black population estimate least trustworthy, and what makes you say so? And for which groups is the estimate untrustworthy in every tract?

When you combine small categories (Lab 3 combined American Indian and Pacific Islander into one indigenous category), you are making a choice about who becomes visible and who disappears into "other." Make the choice deliberately, and write it in your Data Notes.

## 4. Shares, tract by tract

Same as Lab 3, section 5. A `for` loop saves us from writing the division eight times.

In [ ]:
# groups was defined in section 3: all eight categories
for g in groups:
    df_tracts[f'pct_{g}'] = df_tracts[g] / df_tracts['total'] * 100

share_cols = ['NAME'] + [f'pct_{g}' for g in groups]
df_tracts[share_cols].round(1)

A `NaN` in a row means that tract's total is zero (the port and Army Base tracts). Each of the other percentages is an estimate divided by another estimate, so each one carries a margin of error we have not calculated. That is deliberate. Wednesday the 23rd we put the error bars back on, properly.

## 5. The city and the county

The comparison geographies for Assignment 1. Same calls as Lab 3, sections 3.2 and 3.3, wrapped in a small function so we can reuse it for a different year in a minute.

In [ ]:
def pull(geo, year):
    """Pull B03002 for one geography dictionary and one ACS year, cleaned and with shares."""
    df = pd.DataFrame(
        c.acs5.get(list(variables_of_interest.keys()), geo, year=year)
    ).rename(columns=variables_of_interest)
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col])
    # Controlled totals come back as -555555555 (Lab 3, section 3.4). Treat as zero MOE.
    df = df.replace(-555555555, 0)
    for g in groups:
        df[f'pct_{g}'] = df[g] / df['total'] * 100
    df['year'] = year
    return df

df_city   = pull({'for': 'place:53000', 'in': 'state:06'}, ACS_YEAR)   # Oakland
df_county = pull({'for': 'county:001', 'in': 'state:06'}, ACS_YEAR)     # Alameda County

df_city[share_cols].round(1)

### 5.1 Tracts, city, county in one table

We have three DataFrames with the same columns: thirteen tract rows, one Oakland row, one Alameda County row. `pd.concat` stacks DataFrames on top of each other, matching columns by name, so the result is one table with fifteen rows.

Reading the line of code from the inside out:

* `df_tracts[share_cols]` keeps only the columns in `share_cols` (NAME and the eight `pct_` columns). Same for the city and county frames. Cutting each one down to the same columns first is what makes the rows line up.
* The square brackets around the three frames make a Python list; `pd.concat` takes a list of DataFrames and stacks them in the order given: tracts, then city, then county.
* `ignore_index=True` renumbers the rows 0 to 14. Without it each frame keeps its own row numbers, so you would see 0, 1, 2 ... then 0 again, which is confusing and can cause errors later.

If you want the general reference, the pandas user guide page "Merge, join, concatenate and compare" covers `concat` (stacking) and `merge` (joining on a shared column, which you will meet in Lab 4).

In [ ]:
df_compare = pd.concat(
    [df_tracts[share_cols], df_city[share_cols], df_county[share_cols]],
    ignore_index=True
)
df_compare.round(1)

This is the shape of an Assignment 1 exhibit: your neighborhood's tracts against the city and the county. Read across a row and down a column. Which tracts look like Oakland? Which look nothing like it?

## 6. Change over time

Assignment 1 asks how conditions have changed. In ACS terms that means the **same table, two five-year periods that do not overlap**. The most recent period is 2020 to 2024. The most recent period that does not overlap it is 2015 to 2019, which the API calls `year=2019`.

### 6.1 City and county, five years earlier

In [ ]:
EARLIER_YEAR = 2019   # 2015 to 2019 5-year estimates

df_city_2019   = pull({'for': 'place:53000', 'in': 'state:06'}, EARLIER_YEAR)
df_county_2019 = pull({'for': 'county:001', 'in': 'state:06'}, EARLIER_YEAR)

df_change = pd.concat([df_city_2019, df_city, df_county_2019, df_county])
df_change[['NAME', 'year'] + [f'pct_{g}' for g in groups]].round(1)

Two rows per place, five years apart. Those differences are what you will describe in Assignment 1. Whether a difference is real or within the margin of error is a question for the week of September 28.

### 6.2 The same tracts, five years earlier

Here is where it gets interesting. Census tract boundaries were redrawn after the 2020 Census. The 2015 to 2019 estimates sit on the **2010** tract boundaries; the 2020 to 2024 estimates sit on the **2020** boundaries. Most tracts kept their numbers. Some were split, merged, or renumbered.

So we pull all tracts again for 2019 and ask for the same list. Watch the "missing" line.

In [ ]:
df_all_tracts_2019 = pd.DataFrame(
    c.acs5.get(
        list(variables_of_interest.keys()),
        {'for': 'tract:*', 'in': 'state:06 county:001'},
        year=EARLIER_YEAR
    )
).rename(columns=variables_of_interest)

df_tracts_2019 = df_all_tracts_2019[df_all_tracts_2019['tract'].isin(WEST_OAKLAND_TRACTS)].copy()
for col in numeric_cols:
    df_tracts_2019[col] = pd.to_numeric(df_tracts_2019[col])
for g in groups:
    df_tracts_2019[f'pct_{g}'] = df_tracts_2019[g] / df_tracts_2019['total'] * 100

found_2019 = set(df_tracts_2019['tract'])
missing_2019 = [t for t in WEST_OAKLAND_TRACTS if t not in found_2019]
print(f'2015 to 2019: found {len(found_2019)} of {len(WEST_OAKLAND_TRACTS)} tracts.')
print('Missing in 2019:', missing_2019 if missing_2019 else 'none')

df_tracts_2019[share_cols].round(1)

If a tract is missing in one period, its boundary changed, and you cannot compare it directly. Three ways to handle that in Assignment 1, from simplest to most work:

1. Choose tracts whose numbers appear in both periods
2. Describe change at the city and county level, and use tracts for the current period only
3. Use a crosswalk file that reallocates the old tracts onto the new boundaries (optional)

Whichever you do, write it in your Data Notes.

**How a crosswalk works, in three sentences.** A crosswalk is a table with one row for every pair of an old (2010) tract and a new (2020) tract that overlap, and a weight saying what share of the old tract's people (or housing units) fell inside the new tract, estimated from 2020 block counts. To move 2015 to 2019 counts onto 2020 boundaries, you merge your 2019 tract table with the crosswalk on the old tract ID, multiply each count by the weight, then group by the new tract ID and sum. That is three pandas operations (`merge`, a multiplication, `groupby().sum()`), and it works for counts; medians cannot be crosswalked this way, and the MOEs of the reallocated counts need their own treatment, so the crosswalk path is for the advanced track.

The Longitudinal Tract Database at Brown University publishes a 2010 to 2020 crosswalk (NHGIS publishes one too). Professor Acey's 2021 notebook that crosswalks 2000 tracts onto 2010 tracts is the worked example; the 2020 version is the same code with a different file.

## 7. The step we are not taking today

You probably want one number for West Oakland, not ten rows. To get it you add the ten tract estimates together, which is easy, and you combine the ten margins of error, which is **not** simple addition. Adding margins of error overstates the uncertainty, because a bigger sample is more precise, not less.

There is a formula for it, and it is the whole point of next week: Monday the 21st is where margins of error come from (sampling), Wednesday the 23rd is how to combine them, and Lab 4 that week is you doing it for your own tracts. Until then, we leave the tracts as rows.

## 8. Save and finish

In [ ]:
df_tracts.to_csv('west_oakland_tracts_2024.csv', index=False)
df_tracts_2019.to_csv('west_oakland_tracts_2019.csv', index=False)
df_change.to_csv('oakland_alameda_2019_2024.csv', index=False)
print('Saved.')

## Appendix, if you want more: the API without the package

Everything above went through the `census` package. The package is a convenience: it takes the four things we told it (which survey, which variables, which geography, which year), turns them into a web address, adds your key, sends the request, and hands back something pandas can read. You will not need anything else for Assignment 1.

It is worth seeing the web address once, for three reasons:

* **Some data are only reachable this way.** The package knows the ACS and the decennial census. It does not know the ACS microdata sample (PUMS), the population estimates program, county business patterns, or the city and state open data portals you will meet in the October 7 lecture. Those all work the same way as the address below.
* **The package can lag a new release.** When a new vintage of the ACS comes out, the web address works that day; the package may need an update first.
* **It shows you what was happening.** Every data API you use for the rest of your career is a web address with pieces in it. Once you can read this one, you can read the others.

Here is the anatomy of a request, one piece at a time:

| Piece | What it says | The package's version |
| --- | --- | --- |
| `https://api.census.gov/data/2024/acs/acs5` | the dataset: 2024 vintage, ACS, 5-year estimates | `c.acs5` and `year=2024` |
| `?get=NAME,B03002_001E,B03002_001M,...` | the variables you want, separated by commas | the list of variable codes |
| `&for=tract:*` | the geography level, and which ones (`*` means all) | the `for` argument |
| `&in=state:06%20county:001` | the larger geography that contains them (`%20` is a space) | the `in` argument |
| `&key=...` | your API key | added for you |

The cells below build that address, send it with `requests` (the general-purpose Python tool for talking to web services), and turn the answer into a DataFrame by hand.

In [ ]:
import requests
import os
import pandas as pd

# This appendix stands on its own: if you are running only these cells (for example
# after the kernel restarted), it needs the key and the year again.
with open(os.path.expanduser('~/census_key.txt')) as f:
    api_key = f.read().strip()
ACS_YEAR = 2024
WEST_OAKLAND_TRACTS = ['401400', '401500', '401600', '401700', '401800',
                       '402200', '402400', '402500', '402600', '402700',
                       '410500', '981900', '982000']

# The URL says everything the package was saying for us:
# dataset / variables / geography / year, plus the key
url = (f'https://api.census.gov/data/{ACS_YEAR}/acs/acs5'
       f'?get=NAME,B03002_001E,B03002_001M,B03002_004E,B03002_004M,B03002_009E,B03002_009M'
       f'&for=tract:*&in=state:06%20county:001&key={api_key}')

response = requests.get(url)
raw = response.json()      # a list of lists; the first list is the column names
raw[:3]

In [ ]:
# Build the DataFrame yourself: first row is the header, the rest are the data
df_raw = pd.DataFrame(raw[1:], columns=raw[0])
df_raw = df_raw.rename(columns={'B03002_001E': 'total', 'B03002_001M': 'total_moe',
                                'B03002_004E': 'nh_black', 'B03002_004M': 'nh_black_moe',
                                'B03002_009E': 'nh_multi', 'B03002_009M': 'nh_multi_moe'})
for col in ['total', 'total_moe', 'nh_black', 'nh_black_moe', 'nh_multi', 'nh_multi_moe']:
    df_raw[col] = pd.to_numeric(df_raw[col])

df_raw[df_raw['tract'].isin(WEST_OAKLAND_TRACTS)]

In [ ]:
# One thing the raw API does that the package does not: pull an entire table at once
# with group(), instead of listing every variable code. Useful when exploring a table.
url_group = (f'https://api.census.gov/data/{ACS_YEAR}/acs/acs5'
             f'?get=group(B03002)&for=county:001&in=state:06&key={api_key}')
whole_table = requests.get(url_group).json()
print(f'{len(whole_table[0])} columns came back for Alameda County')
whole_table[0][:12]   # the first dozen column names

## What this means for P/NP #4 (due Wednesday, September 23)

West Oakland was today's example. Your case study can be West Oakland, another tour neighborhood, or somewhere else; Lab 4 uses whatever tracts you choose. Your P/NP #4 needs the same ingredients this notebook used:

* Your neighborhood, named, and the tract numbers that make it up (check that they exist in both 2019 and 2024)
* The city or county you will compare it with (or the time frame, if you are comparing across time)
* Your topic, and whether you are taking the new learner path, the advanced path, or a mix
* A table number and title for each of your four key questions in Assignment 1 (plus the optional fifth)

If you can fill in the `WEST_OAKLAND_TRACTS` list and the `variables_of_interest` dictionary for your own place and your own tables, you are ready for Lab 4.